In [295]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=netCDF['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=netCDF.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=netCDF['time']
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF
# data=netCDF.isel(time=np.arange(0,140+1))
# parcel=parcel.isel(time=np.arange(0,140+1))
res='1km'
index_adjust=0
ocean_fraction=0.25


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [ ]:
###################################################################################################################################
#FUNCTIONS

In [290]:
#Get Data Functions
def get_2dtime_data(data,varname,tlev,zlev):
    cloud_var=data[varname].isel(time=tlev,zh=zlev).values
    return cloud_var
def get_3dtime_data(data,varname,tlev):
    cloud_var=data[varname].isel(time=tlev).values
    return cloud_var

def get_conv(t):
    import h5py
    # print('calculating convergence and taking mean')
    dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
    file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{Np_str}' + '.h5'
    with h5py.File(file_path, 'r') as f:
        Conv = f['conv'][t]
    return Conv

In [291]:
#TIME 00:00:00 Function
#Gets the realtime for the current timestep
def get_time(t):
    # init_day,init_hour,init_min=0,0,0
    init_day,init_hour,init_min=0,6,0
    times=netCDF['time'].values/(1e9 * 60); time_inc=times.astype(int)[1]-times.astype(int)[0]
    current_min=init_hour*60+init_min+time_inc*t;
    
    days = init_day + (current_min // (24 * 60))
    
    remain_min = (init_min+time_inc*t) % (24 * 60); 
    hours = (init_hour + (remain_min // 60)) % 24
    mins = remain_min % 60

    ##############################################
    days=str(days);hours=str(hours);mins=str(mins)
    if len(days)==1:days='0'+days
    if len(hours)==1:hours='0'+hours
    if len(mins)==1:mins='0'+mins
    ##############################################

    combo=days+":"+hours+":"+mins
    return days,hours,mins

In [292]:
#Function for taking x and y derivatives (Gradient)
def cd2d(f,dx,dy): #size not compatible, cant calculate adjacent gradient
    ddx = (
            f[:,:, 1:  ]
            -
            f[:,:, 0:-1]
        ) / (
        2 * dx
    )
    
    ddy = (
        f[:,1:, :]
        -
        f[:,0:-1, :]
    ) / (
        2 * dy
    )
    
    return ddx, ddy

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr; import time as time
start_time=time.time()
folder_path = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/trackout/CL_tracking_plots/' #*** plot image output foldername
import os; os.makedirs(folder_path, exist_ok=True)
####################################################################################################################################
def user_deletion():
    #reads user input for timesteps to forget as instructed by teh user in the usersbz.txt file
    filename='usersbz_1km.txt' #*** timestep deletion filename
    # filename='usersbz_250m.txt' #*** timestep deletion filename
    a,b,c,d=[],[],[],[]
    try:
        with open(folder_path+filename, 'r') as file: #*switch to lbz for land breezes 
            for line in file:
                line = line.strip()  # Remove leading/trailing whitespace
                if ',' in line:
                    if '-' not in line: #single case correction
                        # For rows with commas, split and append to a, b, and c lists
                        values = line.split(',')
                        a.append(int(values[0]))
                        b.append(int(values[1]))
                        c.append(int(values[2]))
                    elif '-' in line: #multiple case correction
                        values = line.split(',')   
                        start, end = map(int, values[0].split('-'))
                        a.extend(range(start, end + 1))
                        b=b+[int(values[1])]*((end-start)+1)
                        c=c+[int(values[2])]*((end-start)+1)
                elif '-' in line and ',' not in line: #ignore these
                    # For rows with hyphen, append the range to d list
                    start, end = map(int, line.split('-'))
                    d.extend(range(start, end + 1))
    
    except FileNotFoundError:
        print(f'The file {filename} does not exist. Add if needed.')
    forget = d #needed for single and all max algorithm
    return forget
####################################################################################################################################

In [269]:
def find_SBZ_xmaxs():
    # Define the directory and file path
    dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
    file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{Np_str}' + '.h5'
    
    # Open the HDF5 file in read mode
    with h5py.File(file_path, 'r') as f:
        # Access the 'conv' dataset
        conv_dataset = f['conv']
        
        # Define the vertical level you are interested in
        zlev = 4
        
        # Initialize a list to store the xmaxs for each time step
        xmaxs_list = []

        # Loop over each time step (axis=0 corresponds to time)
        for t in range(conv_dataset.shape[0]):  # conv_dataset.shape[0] is the time dimension size
            # Read the relevant slice for this time step and vertical level
            Conv_t_zlev = conv_dataset[t, zlev, :, :]  # Shape should be (y_size, x_size)
            
            # Calculate the mean across the y-axis
            Conv_ymean = np.mean(Conv_t_zlev, axis=0)  # Mean across the y-axis
            
            # Find the index of the maximum value along the x-axis
            xmax = np.argmax(Conv_ymean)
            
            # Append the result for this time step
            xmaxs_list.append(xmax)
    
    # Convert the list of xmaxs to a numpy array (optional)
    xmaxs = np.array(xmaxs_list)

    return xmaxs #returns SBZ x location for each timestep
SBZ_MAXS=find_SBZ_xmaxs()
print('done')

done


In [288]:
#Finds all local maximums (from Calculus) along each y level for a specific z level (~0.28km in this case)
def find_local_maxes(conv_z,t,yind,conv_thresh,ONLY_SBZ):
    yconv=conv_z[yind,:]
    xf=data['xf'].values.astype(int);
    kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #finds how many x grids is 1 km
    dx=np.round(data['xf'][1]-data['xf'][0],1).item()
    
    #takes dconv/dx
    f=yconv
    ddx = (
            f[1:  ]
            -
            f[0:-1]
        ) / (
        2 * dx
    )

    if np.any(np.isin(forget,t)): 
        local_maxes=np.full_like(local_maxes,np.nan,dtype=float); #forget timestep if in forget variable
    else:
        #finds local max where dconv/dx sign changes
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here (it corrects the location of the derivative)
        local_maxes=local_maxes[np.where(yconv[local_maxes]>conv_thresh)] #check if convergence is greater than convergence threshold (0.9s-1)
        local_maxes=local_maxes[(local_maxes>50*kms)&(local_maxes<len(xf)-50*kms)] #removes maxes that are with 50 km of y boundary
        local_maxes=local_maxes[local_maxes>int(len(xf)*ocean_fraction)] #restricts to right land side
        if ONLY_SBZ==True:
            local_maxes=local_maxes[(local_maxes>=SBZ_MAXS[t]-10*kms)&(local_maxes<=SBZ_MAXS[t]+10*kms)] #removes maxes that are with 50 km of y boundary
            # print(SBZ_MAXS[t]) #TESTING
        # ################################################################################
        # #second round maxes (not 100% necessary, only if missing many convergence maximums that are visually there)
        # yconv2=yconv.copy()
        # yconv2[local_maxes]=0
        # #takes dconv/dx
        # f=yconv2
        # ddx = (
        #         f[1:  ]
        #         -
        #         f[0:-1]
        #     ) / (
        #     2 * dx
        # )
        # signs = np.sign(ddx)
        # signs_diff=np.diff(signs)
        # local_maxes2=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        # local_maxes2=local_maxes2[np.where(yconv2[local_maxes2]>conv_thresh)] #remove local maxes less than zero
        # local_maxes2=local_maxes2[(local_maxes2>50*kms)&(local_maxes2<len(xf)-50*kms)] #removes maxes that are with 50 km of y boundary
        # local_maxes2=local_maxes2[local_maxes2>int(len(xf)/2)] #restricts to right land side
        # local_maxes=np.concatenate((local_maxes,local_maxes2))
        # ################################################################################
    return ddx,local_maxes


In [10]:
###################################################################################################################################
#Imaging Run
# Running Initial Algorithm that Outputs Plots to Remove False Timesteps

In [ ]:
#Find_Local_Maxes Function
#(1) At a single time and z level, runs through each y-level
#(2) At each y-level, takes the x-derivative
#(3) Take sign(x_derivative)
#(4) Take diff(x_derivative)
#(5) Max is located one index to the right of where derivative changes from positive to negative or diff is +1
#[(6) Optional: the algorithm can run a second time over the leftover maxes after removing previous maxes from temporary variable]

In [312]:
#Finding Convergence Lines (CLs) 
####################################################################################################################################
forget=user_deletion() #get forget variable for user specified timesteps to forget
####################################################################################################################################

# #GETTING VMIN and VMAX for Plotting
# dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{Np_str}' + '.h5'
# with h5py.File(file_path, 'r') as f:
#     vmin = f['conv'][:].min()*1000
#     vmax = f['conv'][:].max()*1000

#For each timestep, take the convergence of the field and apply FindLocalMaxes function
# for t in range(0,len(data['time'])): 
def CL_plotting(t,ONLY_SBZ):
    if np.mod(t,10)==0: print(f'current time step: {t}/{len(data["time"])}')

    #Taking Convergence of current timestep
    dx=np.round(data['xf'][1]-data['xf'][0],2).item();dy=dx #gets the dx,dy in meters (0.25m in this case)
    # u=get_3dtime_data(data,'u',t)       
    # v=get_3dtime_data(data,'v',t)  
    # [dudx,dudy]=cd2d(u,dx,dy)
    # [dvdx,dvdy]=cd2d(v,dx,dy)
    # conv = -(dudx + dvdy)
    conv=get_conv(t)

    #Plotting horizontal layer at ~0.28 km 
    zlev=3 #(zf level ~0.28 km)
    channel_aspect_ratio = 5
    figwidth=20
    plt.figure(figsize=(figwidth, figwidth/channel_aspect_ratio)) 
    contour=plt.contourf(conv[zlev]*1000)#,levels=np.arange(-1.5,1.5,.01),vmin=vmin,vmax=vmax)
    
    colorbar = plt.colorbar(contour,label=f'{dx*1000:.0f}'+r'$\ s^{-1}$')# ,pad=pad)
    #####################################################################
    #Gets time for Plot Label
    [days,hours,mins]=get_time(t+index_adjust)
    value=data['zf'][zlev].values; 
    plt.title(f'Convergence at t={t+index_adjust}={days}:{hours}:{mins}, z={value:.2f} km')
    plt.xlabel('x (km)')
    plt.ylabel('y (km)')
    #####################################################################

    # plt.contourf(data['pwat'].isel(time=t),alpha=0.3,cmap='Reds') #rain, prate, pwat #TESTING

    #ADDING TRACKED LOCATION SCATTER
    #for each ylevel, apply FindLocalMaxes Function
    zlev=3;conv_z=conv[zlev,:,:] #only look at 3rd level of convergence field
    maxconv_x=np.array([],dtype='int'); 
    for yind in range(0,len(data['yh'])): #plot maximums for each row
        if res=='1km':
            [ddx,local_maxes]=find_local_maxes(conv_z,t,yind,0.9/1000,ONLY_SBZ) #finds all local maxes with threshold 1km:0.9 250m:3.0
        elif res=='250m':
            [ddx,local_maxes]=find_local_maxes(conv_z,t,yind,3.0/1000,ONLY_SBZ) #finds all local maxes with threshold 1km:0.9 250m:3.0
        plt.scatter(local_maxes,np.array([yind]*len(local_maxes)),color='red',s=5)

    # SAVING PLOT
    if ONLY_SBZ==False:
        plt.savefig(os.path.join(folder_path, f"ALL_CLS/plot_{t+index_adjust}.png"),dpi=72) #save the plot
    if ONLY_SBZ==True:
        plt.savefig(os.path.join(folder_path, f"ONLY_SBZS/plot_{t+index_adjust}.png"),dpi=72) #save the plot
    print('finished saving plot') #TESTING
    plt.close()

for t in range(t,len(data['time'])):
    ONLY_SBZ=False
    # ONLY_SBZ=True
    CL_plotting(t,ONLY_SBZ)
    
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds")

finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
current time step: 25/133
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
finished saving plot
current 

In [ ]:
###################################################################################################################################
#Make Animation

In [ ]:
#Make a GIF of all Tracked Images
from PIL import Image
import os

ONLY_SBZ=False
# ONLY_SBZ=True
# Folder containing the pictures
if ONLY_SBZ==False:
    input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/trackout/CL_tracking_plots/ALL_CLS/'
if ONLY_SBZ==True:
    input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/trackout/CL_tracking_plots/ONLY_SBZS/'

image_filenames = sorted([filename for filename in os.listdir(input_folder) if filename.endswith((".jpg", ".png"))], 
                         key=lambda x: int(x.split('_')[-1].split('.')[0]))

# List to store image objects
images = []

# Iterate over all files in the folder
for filename in image_filenames:
    # Check if the file is an image
    if (filename.endswith(".jpg") or filename.endswith(".png")):
        # Open the image and append it to the list
        images.append(Image.open(os.path.join(input_folder, filename)))

# Save images as a GIF
output_path = input_folder+f'CL_plots_{res}.gif'
images[0].save(output_path,
               save_all=True,
               append_images=images[1:],
               duration=100,  # Specify duration for each frame in milliseconds
               loop=0)

In [314]:
def convert_gif_to_mp4(input_file, output_file, fps,speed,bitrate='750k'):
    from moviepy.editor import VideoFileClip, vfx
    # Load the GIF file
    gif_clip = VideoFileClip(input_file)

    # Set the desired framerate if provided
    if fps:
        gif_clip = gif_clip.set_fps(fps)
    if speed != 1.0:
        gif_clip = gif_clip.fx(vfx.speedx, speed)

    # Write the GIF as an MP4 file
    gif_clip.write_videofile(output_file, codec="libx264",bitrate=bitrate)


ONLY_SBZ=False
# ONLY_SBZ=True
input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/trackout/CL_tracking_plots/'
if ONLY_SBZ==False:
    input_folder += 'ALL_CLS/'
elif ONLY_SBZ==True:
    input_folder += 'ONLY_SBZS/'
input_file =  input_folder + f'CL_plots_{res}.gif'


if ONLY_SBZ==False:
    output_file = f'/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_{res}_{Np_str}_ALL_CLS.mp4'
if ONLY_SBZ==True:
    output_file = f'/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_{res}_{Np_str}_ONLY_SBZS.mp4'
    
convert_gif_to_mp4(input_file, output_file, speed=0.25,fps=None)

Moviepy - Building video /mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_1km_1e6_ONLY_SBZS.mp4.
Moviepy - Writing video /mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_1km_1e6_ONLY_SBZS.mp4



Moviepy - Done !
Moviepy - video ready /mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/CL_plots_1km_1e6_ONLY_SBZS.mp4


In [ ]:
###################################################################################################################################
#Calculation Run

In [217]:
#Job Array

num_jobs=30 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***

job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
if job_id==0: job_id=1
num_parcels=len(data['time']) #total number of parcels
job_range = num_parcels//num_jobs #number of parcels per job 

# Calculate start and end based on job_id
start_job = (job_id - 1) * job_range
end_job = start_job + job_range
if job_id==num_jobs: end_job=num_parcels-1
print(f'running for timesteps {start_job}-{end_job-1}')

data=data.isel(time=slice(start_job,end_job))
index_adjust = int(data['time'][0].values/1e9/60)

running for timesteps 0-3


In [228]:
#SBZ Convergence Line Search Algorithm (levels are seperate) (python version 3.10.9) (All Max Algorithm)
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr; import time as time
start_time=time.time()
####################################################################################################################################
forget=user_deletion() #get forget variable for user specified timesteps to forget
####################################################################################################################################

def layermax(t,ONLY_SBZ): #finds max convergence along y for multiple z location (5 is good)
    num_zlevs=6
    maxconv_x=np.full((num_zlevs+1,len(data['yh']),len(data['xh'])), -1, dtype=int)
    #running again for all levels 
    for zlev in range(0,num_zlevs+1):
        #Taking Convergence of current timestep
        dx=np.round(data['xf'][1]-data['xf'][0],2).item();dy=dx #gets the dx,dy in meters (0.25m in this case)
        conv=get_conv(t)
    
        conv_z=conv[zlev,:,:] #current z level for convergence

        for yind in range(0,len(data['yh'])): #plot maximums for each row
            #finds all local maxes
            if res=='1km':
                [ddx,local_maxes]=find_local_maxes(conv_z,t,yind,0.9/1000,ONLY_SBZ) #finds all local maxes with threshold 1km:0.9 250m:3.0
            elif res=='250m':
                [ddx,local_maxes]=find_local_maxes(conv_zt,yind,3.0/1000,ONLY_SBZ) #finds all local maxes with threshold 1km:0.9 250m:3.0
            
            #storing data
            maxconv_x[zlev,yind,local_maxes] = local_maxes
    return maxconv_x

maxconv_x=layermax(0)
ds1= xr.Dataset({'maxconv_x': (['z','y','x'], maxconv_x)})
for t in range(1,len(data['time'])): #starts from timestep 1 to end
    if np.mod(t,25)==0: print(f'current time step: {t}/{len(data["time"])}')

    ONLY_SBZ=False
    # ONLY_SBZ=True
    maxconv_x=layermax(t,ONLY_SBZ)
    ds2= xr.Dataset({'maxconv_x': (['z','y','x'], maxconv_x)})
    ds1=xr.concat([ds1, ds2], dim='time')


ONLY_SBZ=False
# ONLY_SBZ=True
output_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/trackout/CL_tracking_plots/'
if ONLY_SBZ==False:
    output_folder += 'ALL_CLS/'
elif ONLY_SBZ==True:
    output_folder += 'ONLY_SBZS'
    
ds1.to_netcdf(output_folder+f'whereCL_{job_id}.nc')  

end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds")
#2d5min9mins

Total Elapsed Time: 2.497258186340332 seconds


In [ ]:
#######################################################################################
#Run after finishing job_array section

In [4]:
#another method for concatenation
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr; import time as time
import os

ONLY_SBZ=False
# ONLY_SBZ=True
input_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/trackout/CL_tracking_plots/'
if ONLY_SBZ==False:
    output_folder += 'ALL_CLS/'
elif ONLY_SBZ==True:
    output_folder += 'ONLY_SBZS'

output_folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/'


#Combining all job array outputs
num_jobs=30

job_id=1
whereCL=xr.open_dataset(dir+f'tracking_algorithms/SBZ_plots_{res}/'+f'whereCL_{job_id}.nc')  


dims = whereCL.dims
time_dim = len(data['time'])
z_dim = dims['z']
y_dim = dims['y']
x_dim = dims['x']
maxconv_x = xr.DataArray(
    np.empty((time_dim, z_dim, y_dim, x_dim), dtype=np.int64),
    dims=('time', 'z', 'y', 'x'),
    coords={
        'time': np.arange(time_dim),  # You can customize this to specific time values
        'z': np.arange(z_dim),
        'y': np.arange(y_dim),
        'x': np.arange(x_dim)
    }
)

# Create a new Dataset and add the DataArray as a data variable
new_whereCL = xr.Dataset({'maxconv_x': maxconv_x})

for job_id in np.arange(1,num_jobs+1):
    print(f'current job id: {job_id}')
    whereCL=xr.open_dataset(dir+f'tracking_algorithms/SBZ_plots_{res}/'+f'whereCL_{job_id}.nc')  
    
    num_parcels=401#len(data['time']) #total number of parcels
    job_range = num_parcels//num_jobs #number of parcels per job 
    start_job = (job_id - 1) * job_range
    end_job = start_job + job_range
    if job_id==num_jobs: end_job=num_parcels-1
    
    new_whereCL['maxconv_x'].loc[dict(time=slice(start_job, end_job-1))]=whereCL['maxconv_x'].isel(time=slice(0, end_job-start_job))
    del whereCL

print('saving output') #this part can take quite long
new_whereCL.to_netcdf(output_folder+f'SBZ_plots_{res}/'+f'whereCL_{res}.nc') 
print('done')

current job id: 1
current job id: 2
current job id: 3
current job id: 4
current job id: 5
current job id: 6
current job id: 7
current job id: 8
current job id: 9
current job id: 10
current job id: 11
current job id: 12
current job id: 13
current job id: 14
current job id: 15
current job id: 16
current job id: 17
current job id: 18
current job id: 19
current job id: 20
current job id: 21
current job id: 22
current job id: 23
current job id: 24
current job id: 25
current job id: 26
current job id: 27
current job id: 28
current job id: 29
current job id: 30
done


In [ ]:
################################################################
#TESTING

In [ ]:
# #TESTING OUTPUT *****
# # NEED TO MAKE ABSOLUTE SURE OUTPUT MATCHES PLOT


# #Loading Libraries
# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr; import time as time
# import os
# # pip install h5netcdf; pip install netCDF4

# #Get Data Functions

# def get_2dtime_data(data,varname,tlev,zlev):
#     cloud_var=data[varname].isel(time=tlev,zh=zlev).values
#     return cloud_var
# def get_3dtime_data(data,varname,tlev):
#     cloud_var=data[varname].isel(time=tlev).values
#     return cloud_var

# # function for taking x and y derivatives (Gradient)
# def cd2d(f,dx,dy): 
#     ddx = (
#             f[:,:, 1:  ]
#             -
#             f[:,:, 0:-1]
#         ) / (
#         2 * dx
#     )
    
#     ddy = (
#         f[:,1:, :]
#         -
#         f[:,0:-1, :]
#     ) / (
#         2 * dy
#     )
    
#     return ddx, ddy #should add an option here to restrict to only a certain level to save time*******

# #Getting 250m Data
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc')
# data2=data.copy()

# #Restricts the timesteps of the data from timesteps 0 to 350
# data=data.isel(time=np.arange(400+1))

# #Taking Convergence of current timestep
# dx=np.round(data['xf'][1]-data['xf'][0],2).item();dy=dx #gets the dx,dy in meters (0.25m in this case)
# # u=get_3dtime_data(data,'u',t)       
# # v=get_3dtime_data(data,'v',t)  
# # [dudx,dudy]=cd2d(u,dx,dy)
# # [dvdx,dvdy]=cd2d(v,dx,dy)
# # conv = -(dudx + dvdy)
# conv=get_conv(t) #***

# #Plotting horizontal layer at ~0.28 km 
# zlev=3 #(zf level ~0.28 km)
# t=294
# channel_aspect_ratio = 5
# figwidth=20
# plt.figure(figsize=(figwidth, figwidth/channel_aspect_ratio)) 
# contour=plt.contourf(conv[zlev,:,:],levels=np.arange(-1.5,1.5,.01))
# colorbar = plt.colorbar(contour,label=f'{dx*1000:.0f}'+r'$\ s^{-1}$')# ,pad=pad)


# #Plotting Tracked CL Points
# whereCL=xr.open_dataset(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_250m.nc') 

# for yind in range(0,len(data['yh'])): #plot maximums for each row
#     local_maxes=whereCL['maxconv_x'].isel(time=t,z=3,y=yind).values
    
#     local_maxes = local_maxes.astype(float)
#     local_maxes[local_maxes == -1] = np.nan
#     valid_mask = ~np.isnan(local_maxes)
#     plt.scatter(local_maxes[valid_mask], np.array([yind] * np.sum(valid_mask)), color='red', s=5)

In [ ]:
#extra tests

# #makes sure there is maxes on either side (not robust enough)
# maxes=[find_maxes(conv_sub,ylev,[],i) for i in range(3)]
# if np.all(~np.isin(maxes,np.array([xmax-1,xmax+1]))):
#     xmax=np.nan
# else:
#     #picks second most max if theta within +-3 of max differs by 1 degree #(not robust enough)
#     before=data['th'].isel(time=t,zh=2,yh=ylev,xh=xmax-3).values
#     after=data['th'].isel(time=t,zh=2,yh=ylev,xh=np.mod(xmax+3,len(data['xh']))).values
#     if abs(after-before)<0.7:
#         xmax=np.nan

# xmax=460
# th1=data['th'].isel(time=100,zh=2,yh=ylev,xh=xmax-3).values
# prs=data['prs'].isel(time=100,zh=2,yh=ylev,xh=xmax-3).values
# prs0=data['prs'].isel(time=100,zh=0,yh=ylev,xh=xmax-3).values
# T1=th1/((prs0/prs)**0.286)
# mean = np.mean(maxconv_x)
# sdev = np.sqrt(np.var(maxconv_x))
# thresh=sdev
# [point for point in maxconv_x if abs(point - mean) > thresh] #points to remove

#boundary layer: average vertical profile of clw mixing ratio, lowest layer where clw is lower than 10e-6 kg/kg
# from scipy.stats import chisquare   
# if chisquare(maxconv_x)[1]<0.05: #if pvalue>0.05, data is not distributed. if pvalue<0.05, data is uniformly distributed. (null hypothesis: uniformly distributed)
#     maxconv_x=np.full_like(maxconv_x, np.nan, dtype=np.float64) #only keeps strong CL cases, but we want the weak CLs in the case of stronger cold pool convergences


In [75]:
# #testing
# t=100
# zlev=2
# dx,dy=1,1 
# # u=get_3dtime_data(data,'u',t)       
# # v=get_3dtime_data(data,'v',t)  
# # [dudx,dudy]=cd2d(u,dx,dy)
# # [dvdx,dvdy]=cd2d(v,dx,dy)
# # conv = -(dudx + dvdy)
# conv=get_conv(t) #***

# channel_aspect_ratio = 5
# plt.figure(figsize=(10, 10/channel_aspect_ratio)) 
# contour=plt.contourf(conv[zlev,:,:],levels=np.arange(-1.5,1.5,.01))
# colorbar = plt.colorbar(contour)
# plt.title(f'Conv at t={t},z={zlev}')

# whereCL=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/whereCL_30min_all.nc')
# maxconv_x=whereCL['maxconv_x'].isel(time=t,z=zlev)
# maxconv_y=whereCL['maxconv_y'].isel(time=t,z=zlev)
# plt.scatter(maxconv_x,maxconv_y,color='red',s=4)
# print(data['zf'])

In [ ]:
# #TESTING***

# data=xr.open_dataset(dir+'cm1out_250m.nc')
# t=200

# dx=np.round(data['xf'][1]-data['xf'][0],2).item();dy=dx #gets the dx,dy in meters (0.25m in this case)
# # u=get_3dtime_data(data,'u',t)       
# # v=get_3dtime_data(data,'v',t)  
# # [dudx,dudy]=cd2d(u,dx,dy)
# # [dvdx,dvdy]=cd2d(v,dx,dy)
# # conv = -(dudx + dvdy)
# conv=get_conv(t) #***

# conv=conv[3]

# plt.contourf(conv)

# # crossthresh=np.where(conv>0.12) #testing threshold
# # for count,position in enumerate(zip(*crossthresh)):
# #     plt.scatter(position[1],position[0],color='red',s=5)

# #     print(position)
# #     if count==100: break

# ncol = conv.shape[1]  # Total number of columns in the array
# # Extract the right side of the array
# halfconv=conv[:,int(ncol/2):]
# convmean=np.mean(halfconv, axis=1)
# # plt.plot(convmean) #use 0.12


# halfconv=conv[200,int(ncol/2):]
# plt.plot(halfconv)
# halfconv.max()

In [ ]:
#naive method for concatenating eulerian CL tracking job_array outputs

# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr; import time as time
# import os
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# #Combining all job array outputs
# num_jobs=30

# job_id=1
# print(f'current job id: {job_id}')
# whereCL=xr.open_dataset(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_{job_id}.nc')  

# #save whereCL and remove from memory
# whereCL.to_netcdf(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_1-{job_id}_250m.nc')  
# del whereCL

# for job_id in np.arange(2,num_jobs+1):
#     print(f'current job id: {job_id}')
#     whereCL=xr.open_dataset(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_1-{job_id-1}_250m.nc')  
#     add=xr.open_dataset(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_{job_id}.nc')  
#     whereCL=xr.concat([whereCL, add], dim='time')
#     del add
#     whereCL.to_netcdf(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_1-{job_id}_250m.nc') 
#     del whereCL
# print('done')

# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr; import time as time
# import os
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# #Combining all job array outputs
# num_jobs=30

# #if kernal crashes, continue the run here (uncomment)
# dir_path = dir+'tracking_algorithms/SBZ_plots_250m/'
# numbers = []
# for item in os.listdir(dir_path):
#     # Check if the item matches the expected format
#     if item.startswith('whereCL_1-') and item.endswith('_250m.nc'):
#         # Split the name to get the number
#         parts = item.split('-')
#         if len(parts) > 1:
#             number = parts[-1].split('_')[0]  # Get the part after the last hyphen
#             numbers.append(int(number))  # Convert to int and add to the list
# if numbers:
#     largest_number = max(numbers)
#     print(f'The largest number is: {largest_number}')
# else:
#     print('No valid files found.')

# if largest_number != 30:
#     print(f'continuing at job id: {largest_number+1}')
#     for job_id in np.arange(largest_number+1,num_jobs+1):
#         print(f'current job id: {job_id}')
#         whereCL=xr.open_dataset(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_1-{job_id-1}_250m.nc')  
#         add=xr.open_dataset(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_{job_id}.nc')  
#         whereCL=xr.concat([whereCL, add], dim='time')
#         del add
#         whereCL.to_netcdf(dir+'tracking_algorithms/SBZ_plots_250m/'+f'whereCL_1-{job_id}_250m.nc') 
#         del whereCL
# print('done')